A small development repository still needs credentials. An API client needs a
token; an image uploader needs access to an object store. Keeping those values in
project files makes it easy for a repository read, a diagnostic command, or an
accidental commit to expose them.

Atlas puts local secret values in the system keyring and provides four commands
for working with them. On the current development platform, macOS, the intended
backend is Keychain. The project calls this small command group the Vault.

This is the first of two walkthroughs in [Building Atlas](../collections/building-atlas.qmd).
It covers the CLI and its tests. The follow-up,
[Adding audit logs to the Vault](adding-audit-logs-to-the-vault.qmd), adds a local
command history, retention, and coordination between simultaneous processes.
The CLI excerpts here focus on the command behavior; the current source also
contains the audit wrappers explained in that follow-up.

**1. Start with a command contract.** The Vault needs a way to store, retrieve,
remove, and check credentials. Each operation has a defined input and output:

| Command | Input | Result |
| --- | --- | --- |
| `at vault set NAME` | Hidden prompt, or explicit `--stdin` | Store or replace a value. |
| `at vault get NAME` | Secret name | Copy to the clipboard in a terminal, or write to stdout when captured. |
| `at vault rm NAME` | Confirmation, or explicit `--yes` | Remove the local keyring entry. |
| `at vault check` | Names from `.env.example` | Report which entries exist, without displaying their values. |

The distinction between a secret's name and value matters throughout the design.
`API_TOKEN` is a name that can appear in a manifest and a diagnostic message.
The credential stored under that name should not appear there.

The normal data path is short:

```{mermaid}
%%| label: fig-vault-cli-path
%%| fig-cap: "A Vault command names a credential. The keyring backend owns its stored value."
flowchart TD
    User["Terminal or script"] --> Root["at: root Typer app"]
    Root --> Vault["vault: command group"]
    Vault --> Command["set / get / rm / check"]
    Manifest[".env.example: names only"] --> Command
    Command --> Backend["Python keyring backend"]
    Backend --> Keychain["macOS Keychain"]
```

The diagram describes the default local setup. Python keyring selects its
backend, so backend configuration remains part of the environment. The CLI
rejects the disabled and unavailable backends; it does not certify every
third-party backend as secure.

**2. Connect the executable to a Typer command group.** The executable entry in
`pyproject.toml` points to the root app's `main` function:

```toml
[project.scripts]
at = "atlas.cli:main"
```

The root app mounts a second Typer app under the name `vault`:

```python
import typer

from atlas.vault import app as vault_app

app = typer.Typer(
    no_args_is_help=True,
    help="Atlas tools.",
    pretty_exceptions_show_locals=False,
)
app.add_typer(vault_app, name="vault")


def main() -> None:
    app()
```

Disabling local-variable display in pretty tracebacks reduces the chance that a
secret held in a local variable appears in a diagnostic. Expected keyring
failures get their own shorter error path later in the implementation.

The Vault app registers commands with decorators such as
`@app.command("get")`. Their Python function names can stay descriptive while
the public commands stay short. After installing the project dependencies, the
entry point is available through `uv run`:

```sh
uv sync
uv run at vault --help
```

**3. Give all keyring operations one namespace and one error policy.** Keyring
looks up a value using a service name and a username. Atlas uses the constant
`atlas.vault` as the service and the secret's name as the username.

Conceptually, `API_TOKEN` is stored under this pair:

```python
("atlas.vault", "API_TOKEN")
```

That namespace is shared across projects for the current OS user. Changing the
working directory does not create a separate Vault. Project-specific names or
credentials therefore need deliberate naming and permission choices.

The wrapper around keyring calls supplies the service name and sanitizes
expected failures:

```python
def _call_keyring[T](function: Callable[..., T], *args: str) -> T:
    try:
        return function(SERVICE_NAME, *args)
    except Exception as error:
        _exit(f"Keyring operation failed ({type(error).__name__}).")
```

This excerpt uses the generic-function syntax supported by the project's Python
3.13 requirement. `T` preserves the wrapped function's return type: a lookup can
return a string or `None`, while a write can return `None`.

The error message includes the exception's type, rather than its full text.
A backend exception could include a credential in its message. There is no need
to reproduce that message to explain that the operation failed.

**4. Make secret input explicit.** The interactive command prompts for a hidden
value and asks for confirmation:

```sh
uv run at vault set API_TOKEN
```

The prompt configuration in `set_secret()` is:

```python
with warnings.catch_warnings():
    warnings.simplefilter("error", GetPassWarning)
    value = typer.prompt(
        "Secret", hide_input=True, confirmation_prompt=True, err=True
    )
```

The code first requires `sys.stdin.isatty()` to be true. A **TTY** is a terminal
interface; `isatty()` tells the command whether the stream is attached to one.
Captured input in a test runner or a pipe normally does not satisfy that check.

The warning filter treats `GetPassWarning` as an exception. Python uses this
warning when its password-input function cannot guarantee echo-free input. The
Vault catches it and directs the caller to the explicit stdin path.
[Python's password-input documentation](https://docs.python.org/3.13/library/getpass.html).

A script supplies `--stdin` deliberately. For example, an existing credential
file can feed the command without putting the value in its arguments:

```sh
uv run at vault set API_TOKEN --stdin < /path/to/existing-secret
```

The implementation reads stdin and removes one trailing line ending:

```python
value = sys.stdin.read()
value = value.removesuffix("\n").removesuffix("\r")
```

It does not call `strip()`, which would also remove potentially meaningful
spaces. Empty input is rejected. Other whitespace is preserved apart from that
single trailing line ending.

**5. Separate data output from status messages.** Retrieval has two behaviors,
selected by `sys.stdout.isatty()`:

```python
if sys.stdout.isatty():
    _copy_to_clipboard(value)
    typer.echo(f"Copied {name} to clipboard.", err=True)
else:
    sys.stdout.write(value)
```

In a terminal, `pbcopy` receives the value through its stdin and a status message
goes to stderr. When stdout is captured or redirected, stdout contains exactly
the stored value, with no added newline.

This makes the command usable in scripts, but it is also an intentional release
of the credential. A tool that captures stdout can receive the real value. The
Vault does not promise to hide a retrieved credential from the caller.

Exit codes are another part of the command contract:

| Code | Meaning in the current CLI |
| --- | --- |
| `0` | Success, including a user declining deletion. |
| `1` | Missing secret, missing manifest, or an uncaught ordinary failure. |
| `2` | Invalid manifest, invalid input, or a handled keyring/clipboard error. |

Deletion requires confirmation in a terminal. In scripts, `--yes` or `-y` supplies
that confirmation explicitly. Removing a keyring entry removes the local copy;
revoking the credential at its provider is a separate operation.

**6. Keep the manifest about names.** The repository-root `.env.example` declares
expected entries using empty assignments:

```dotenv
# Required secrets
API_TOKEN=
DATABASE_URL=
```

`check` skips blank lines and full-line comments, requires a nonempty name and an
empty value, then looks up the declared names. It reports `[ok]` or `[no]` and a
count. It does not modify the manifest or display stored values.

This is a small parser rather than a shell interpreter. Its path comes from
`ROOT_PATH`, not the caller's current directory. A missing manifest raises
`FileNotFoundError`; a manifest with no names reports `No secrets declared.`
and succeeds.

A mistaken line such as `API_TOKEN=private value` is rejected without printing
the supplied value. Error handling needs that property too: rejecting an input
does not justify echoing its contents.

**7. Replace Keychain in tests.** Tests should be able to create and delete
credentials freely without changing the developer's real keyring. The test
suite uses an in-memory implementation of the keyring interface:

```python
from keyring.backend import KeyringBackend
from keyring.errors import PasswordDeleteError


class LocalKeyring(KeyringBackend):
    priority = 1

    def __init__(self) -> None:
        self.values: dict[tuple[str, str], str] = {}

    def get_password(self, service: str, username: str) -> str | None:
        return self.values.get((service, username))

    def set_password(self, service: str, username: str, password: str) -> None:
        self.values[(service, username)] = password

    def delete_password(self, service: str, username: str) -> None:
        try:
            del self.values[(service, username)]
        except KeyError as error:
            raise PasswordDeleteError from error
```

Each test gets a fresh instance. The dictionary key matches the service/name
pair used by the real library, so tests exercise the namespace and the complete
store/retrieve/remove behavior.

Fixtures connect this substitute to the application:

```python
from unittest.mock import Mock

import pytest
from typer.testing import CliRunner


@pytest.fixture
def runner() -> CliRunner:
    return CliRunner()


@pytest.fixture
def backend() -> LocalKeyring:
    return LocalKeyring()


@pytest.fixture(autouse=True)
def get_backend(monkeypatch, backend) -> Mock:
    replacement = Mock(return_value=backend)
    monkeypatch.setattr("atlas.vault.keyring.get_keyring", replacement)
    return replacement


@pytest.fixture
def manifest(tmp_path, monkeypatch):
    monkeypatch.setattr("atlas.vault.ROOT_PATH", tmp_path)
    return tmp_path / ".env.example"


@pytest.fixture(autouse=True)
def audit_log(tmp_path, monkeypatch):
    monkeypatch.setattr("atlas.vault_audit.ROOT_PATH", tmp_path)
    monkeypatch.delenv("ATLAS_VAULT_SESSION_ID", raising=False)
    return tmp_path / ".atlas" / "logs" / "vault.jsonl"
```

`autouse=True` applies the backend replacement even when a test does not name
that fixture. `tmp_path` supplies a temporary directory for a test's manifest.
`monkeypatch` restores the replaced attributes after the test.
[pytest's monkeypatch documentation](https://docs.pytest.org/en/stable/how-to/monkeypatch.html).

The last fixture isolates the follow-up article's audit log, which is already
present in the current source. It patches `atlas.vault_audit.ROOT_PATH` separately
and clears `ATLAS_VAULT_SESSION_ID`.
Patching the name in one module does not change a constant already imported by
another module.

```{mermaid}
%%| label: fig-vault-cli-test-path
%%| fig-cap: "CLI tests retain argument parsing and application behavior while replacing external state."
flowchart TD
    Test["pytest test"] --> Runner["CliRunner.invoke(app, arguments)"]
    Runner --> CLI["Real Typer command"]
    CLI --> Fake["Fresh LocalKeyring dictionary"]
    CLI --> Manifest["Manifest under tmp_path"]
    CLI --> Result["Exit code, stdout, stderr"]
    Result --> Assertions["Check behavior and secret exclusion"]
```

**8. Test the public CLI, including its output streams.** `CliRunner` invokes the
Typer app with an argument list and optional simulated input. This exercises
command registration and parsing as well as the command body.
[Typer's testing guide](https://typer.tiangolo.com/tutorial/testing/).

The following test is a compact version of the lifecycle covered in
`tests/test_vault.py`. It uses the fixtures above and imports the actual root
app and namespace:

```python
from atlas.cli import app
from atlas.vault import SERVICE_NAME


def test_secret_lifecycle(runner, backend, manifest):
    contents = "API_TOKEN=\n"
    manifest.write_text(contents)

    stored = runner.invoke(
        app, ["vault", "set", "API_TOKEN", "--stdin"],
        input="dummy credential\n",
    )
    assert stored.exit_code == 0
    assert stored.stdout == ""
    assert "dummy credential" not in stored.output
    assert backend.values == {(SERVICE_NAME, "API_TOKEN"): "dummy credential"}

    retrieved = runner.invoke(app, ["vault", "get", "API_TOKEN"])
    assert retrieved.exit_code == 0
    assert retrieved.stdout == "dummy credential"

    checked = runner.invoke(app, ["vault", "check"])
    assert checked.exit_code == 0
    assert "[ok]  API_TOKEN" in checked.stdout
    assert "dummy credential" not in checked.output

    removed = runner.invoke(app, ["vault", "rm", "API_TOKEN", "--yes"])
    assert removed.exit_code == 0
    assert backend.values == {}
    assert manifest.read_text() == contents
```

The retrieval assertion intentionally expects the credential in stdout: captured
output takes the nonterminal branch. The storage and check assertions expect it
to be absent. A blanket assertion that a secret can never appear in output would
contradict `get`'s public behavior.

Checking `result.stdout` separately from `result.stderr` also catches a subtle
regression: a friendly status message written to stdout could corrupt data sent
to another program. In this suite, `result.output` is useful for checking the
combined visible output for accidental disclosure.

**9. Exercise failures as contracts.** The error tests cover behavior the happy
path cannot establish:

| Test case | Property being checked |
| --- | --- |
| Empty or noninteractive input without `--stdin` | Invalid input cannot create a secret. |
| Missing secret | Exit code `1`, with no value on stdout. |
| Invalid manifest | Failure occurs before keyring lookup, without echoing its contents. |
| Missing manifest | The current `FileNotFoundError` behavior is explicit. |
| Disabled or unavailable backend | Failure is reported rather than silently ignored. |
| Unconfirmed noninteractive deletion | The stored entry survives. |
| Backend exception containing a dummy credential | The visible error omits that credential. |

For the last case, the test injects a deliberately unsafe backend message:

```python
def test_backend_error_is_sanitized(runner, backend, monkeypatch):
    monkeypatch.setattr(
        backend,
        "set_password",
        Mock(side_effect=RuntimeError("dummy credential")),
    )

    result = runner.invoke(
        app, ["vault", "set", "TOKEN", "--stdin"],
        input="dummy credential",
    )

    assert result.exit_code == 2
    assert "RuntimeError" in result.output
    assert "dummy credential" not in result.output
```

The message still identifies the failure category. The test establishes that
the backend's arbitrary exception text does not reach the user-facing error.

Interactive branches need a different setup because `CliRunner` is not a real
terminal. The existing test replaces the Vault module's `sys` reference with a
small object whose stdin reports `isatty() == True`, then supplies a mismatched
confirmation followed by matching inputs. It separately declines and accepts
deletion. Another test makes `typer.prompt` raise `GetPassWarning` and verifies
that nothing is stored.

Those tests establish control flow. They do not demonstrate actual terminal echo
behavior, a Keychain authorization dialog, or the contents of the system
clipboard. A manual smoke check with a disposable credential remains useful
when changing those integrations.

**10. Run the checks and keep their scope clear.** From the repository root,
the CLI tests and the full suite can be run separately:

```sh
uv run pytest tests/test_vault.py -q
uv run pytest --cov=atlas --cov-branch --cov-report=term-missing
```

`make doctor` additionally runs `at vault check` against the real local manifest
and keyring. That makes it an environment check as well as a test command; it is
different from the isolated pytest run.

The Vault reduces accidental exposure from project files and provides a
consistent interface for local secrets. It does not restrict a credential's
remote permissions. A B2 application key that can delete objects still has that
authority after retrieval. Keyring also documents that scripts using the same
Python executable can access its macOS entries without another password prompt
under the described access settings.
[Keyring's security considerations](https://keyring.readthedocs.io/en/stable/#security-considerations).

The next change makes supported command use visible. Continue with
[Adding audit logs to the Vault](adding-audit-logs-to-the-vault.qmd) for the event
format, context managers, file locks, and a rolling 30-day history.